In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip -q install shap lime

import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import shap
import lime

from sklearn.metrics import average_precision_score
from sklearn.model_selection import ParameterGrid
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.layers import Input, Embedding, Dense, Flatten, Dropout, concatenate, SpatialDropout1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 19.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
DATA_PATH = "/content/drive/MyDrive/Thesis2/train.csv"

dtypes = {
    "ip": "uint32",
    "app": "uint16",
    "device": "uint16",
    "os": "uint16",
    "channel": "uint16",
    "is_attributed": "uint8"
}

usecols = ["ip", "app", "device", "os", "channel", "click_time", "is_attributed"]

df = pd.read_csv(
    DATA_PATH,
    usecols=usecols,
    dtype=dtypes,
    parse_dates=["click_time"]
)

print(df.shape)
display(df.head())
display(df.dtypes)

(184903890, 7)


,ip,app,device,os,channel,click_time,is_attributed
0,83230,3,1,13,379,2017-11-06 14:32:21,0
1,17357,3,1,19,379,2017-11-06 14:33:34,0
2,35810,3,1,13,379,2017-11-06 14:34:12,0
3,45745,14,1,13,478,2017-11-06 14:34:52,0
4,161007,3,1,13,379,2017-11-06 14:35:08,0


,0
ip,uint32
app,uint16
device,uint16
os,uint16
channel,uint16
click_time,datetime64[ns]
is_attributed,uint8


In [ ]:
TIME_STEP_HOURS = 8

def add_time_steps(data, step_hours=8):
    data = data.copy()
    start_time = data["click_time"].min()
    elapsed_hours = (data["click_time"] - start_time).dt.total_seconds() / 3600.0
    data["time_step"] = (elapsed_hours // step_hours).astype("uint16") + 1
    return data

df = add_time_steps(df, step_hours=TIME_STEP_HOURS)
df = df.sort_values("click_time").reset_index(drop=True)

print(df["time_step"].min(), df["time_step"].max(), df["time_step"].nunique())
display(df[["click_time", "time_step"]].head())

1 10 10


,click_time,time_step
0,2017-11-06 14:32:21,1
1,2017-11-06 14:33:34,1
2,2017-11-06 14:34:12,1
3,2017-11-06 14:34:52,1
4,2017-11-06 14:35:08,1


In [ ]:
L = 2
S = 1
H = 1

def build_window_pairs(data, L=2, S=1, H=1):
    max_step = int(data["time_step"].max())
    pairs = []

    for k in range(L, max_step - S - H + 1):
        pairs.append({
            "k": k,
            "train_A_steps": list(range(k - L + 1, k + 1)),
            "train_B_steps": list(range(k - L + 1 + S, k + 1 + S)),
            "eval_steps": list(range(k + S + 1, k + S + H + 1))
        })

    return pairs

window_pairs = build_window_pairs(df, L=L, S=S, H=H)

print("Number of window pairs:", len(window_pairs))
display(pd.DataFrame(window_pairs).head())

Number of window pairs: 7


,k,train_A_steps,train_B_steps,eval_steps
0,2,"[1, 2]","[2, 3]",[4]
1,3,"[2, 3]","[3, 4]",[5]
2,4,"[3, 4]","[4, 5]",[6]
3,5,"[4, 5]","[5, 6]",[7]
4,6,"[5, 6]","[6, 7]",[8]


In [ ]:
def prepare_window_data(full_df, steps):
    part = full_df[full_df["time_step"].isin(steps)].copy()
    part = part.sort_values("click_time").reset_index(drop=True)

    part["hour"] = part["click_time"].dt.hour.astype("uint8")
    part["day"] = part["click_time"].dt.day.astype("uint8")
    part["wday"] = part["click_time"].dt.dayofweek.astype("uint8")

    part["qty"] = part.groupby(["ip", "day", "hour"])["ip"].transform("size").astype("float32")
    part["ip_app_count"] = part.groupby(["ip", "app"])["ip"].transform("size").astype("float32")
    part["ip_app_os_count"] = part.groupby(["ip", "app", "os"])["ip"].transform("size").astype("float32")
    part["ip_channel_count"] = part.groupby(["ip", "channel"])["ip"].transform("size").astype("float32")
    part["ip_device_count"] = part.groupby(["ip", "device"])["ip"].transform("size").astype("float32")
    part["ip_unique_apps"] = part.groupby(["ip"])["app"].transform("nunique").astype("float32")
    part["ip_unique_channels"] = part.groupby(["ip"])["channel"].transform("nunique").astype("float32")
    part["app_channel_count"] = part.groupby(["app", "channel"])["app"].transform("size").astype("float32")

    part["prev_click_ip"] = part.groupby("ip")["click_time"].shift(1)
    part["prev_click_ip_app"] = part.groupby(["ip", "app"])["click_time"].shift(1)

    part["secs_since_prev_ip"] = (part["click_time"] - part["prev_click_ip"]).dt.total_seconds().fillna(0).astype("float32")
    part["secs_since_prev_ip_app"] = (part["click_time"] - part["prev_click_ip_app"]).dt.total_seconds().fillna(0).astype("float32")

    num_log_cols = [
        "qty", "ip_app_count", "ip_app_os_count", "ip_channel_count", "ip_device_count",
        "ip_unique_apps", "ip_unique_channels", "app_channel_count",
        "secs_since_prev_ip", "secs_since_prev_ip_app"
    ]

    for col in num_log_cols:
        part[col] = np.log1p(part[col]).astype("float32")

    part = part.drop(columns=["prev_click_ip", "prev_click_ip_app"])

    return part

In [ ]:
TARGET_COL = "is_attributed"

CAT_COLS = ["app", "device", "os", "channel", "hour", "day", "wday"]
NUM_COLS = [
    "qty", "ip_app_count", "ip_app_os_count", "ip_channel_count", "ip_device_count",
    "ip_unique_apps", "ip_unique_channels", "app_channel_count",
    "secs_since_prev_ip", "secs_since_prev_ip_app"
]

def get_keras_data(dataset):
    X = {
        "app": np.array(dataset["app"]).astype("int32"),
        "dev": np.array(dataset["device"]).astype("int32"),
        "os": np.array(dataset["os"]).astype("int32"),
        "ch": np.array(dataset["channel"]).astype("int32"),
        "h": np.array(dataset["hour"]).astype("int32"),
        "d": np.array(dataset["day"]).astype("int32"),
        "wd": np.array(dataset["wday"]).astype("int32"),
        "num": np.array(dataset[NUM_COLS]).astype("float32")
    }
    return X

def get_max_values(train_df, val_df=None, eval_df=None):
    frames = [train_df]
    if val_df is not None:
        frames.append(val_df)
    if eval_df is not None:
        frames.append(eval_df)

    merged = pd.concat(frames, axis=0)

    max_vals = {
        "app": int(merged["app"].max()) + 1,
        "dev": int(merged["device"].max()) + 1,
        "os": int(merged["os"].max()) + 1,
        "ch": int(merged["channel"].max()) + 1,
        "h": int(merged["hour"].max()) + 1,
        "d": int(merged["day"].max()) + 1,
        "wd": int(merged["wday"].max()) + 1
    }

    return max_vals

def fit_num_scaler(train_df):
    mu = train_df[NUM_COLS].mean().astype("float32")
    sd = train_df[NUM_COLS].std().replace(0, 1).astype("float32")
    return mu, sd

def apply_num_scaler(df_in, mu, sd):
    df_out = df_in.copy()
    df_out[NUM_COLS] = ((df_out[NUM_COLS] - mu) / sd).astype("float32")
    return df_out

def summarize_positive_counts(name, df_in):
    n = len(df_in)
    pos = int(df_in[TARGET_COL].sum())
    rate = float(df_in[TARGET_COL].mean()) if n > 0 else 0.0
    return {
        "set": name,
        "n_rows": n,
        "n_pos": pos,
        "pos_rate": rate
    }

In [ ]:
def build_embedding_nn(max_vals, emb_n=32, dense_n=256, dropout=0.2, lr=0.001):
    in_app = Input(shape=[1], name="app")
    emb_app = Embedding(max_vals["app"], emb_n)(in_app)

    in_ch = Input(shape=[1], name="ch")
    emb_ch = Embedding(max_vals["ch"], emb_n)(in_ch)

    in_dev = Input(shape=[1], name="dev")
    emb_dev = Embedding(max_vals["dev"], emb_n)(in_dev)

    in_os = Input(shape=[1], name="os")
    emb_os = Embedding(max_vals["os"], emb_n)(in_os)

    in_h = Input(shape=[1], name="h")
    emb_h = Embedding(max_vals["h"], emb_n)(in_h)

    in_d = Input(shape=[1], name="d")
    emb_d = Embedding(max_vals["d"], emb_n)(in_d)

    in_wd = Input(shape=[1], name="wd")
    emb_wd = Embedding(max_vals["wd"], emb_n)(in_wd)

    in_num = Input(shape=(len(NUM_COLS),), name="num")

    fe = concatenate([emb_app, emb_ch, emb_dev, emb_os, emb_h, emb_d, emb_wd])
    x = SpatialDropout1D(dropout)(fe)
    x = Flatten()(x)
    x = concatenate([x, in_num])
    x = Dense(dense_n, activation="relu")(x)
    x = Dropout(dropout)(x)
    x = Dense(dense_n, activation="relu")(x)
    x = Dropout(dropout)(x)
    outp = Dense(1, activation="sigmoid")(x)

    model = Model(
        inputs=[in_app, in_ch, in_dev, in_os, in_h, in_d, in_wd, in_num],
        outputs=outp
    )

    model.compile(
        loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0),
        optimizer=Adam(learning_rate=lr),
        metrics=[tf.keras.metrics.AUC(curve="PR", name="pr_auc")]
    )

    return model

In [ ]:
def split_train_val_last_20_percent(window_df, val_frac=0.2):
    window_df = window_df.sort_values("click_time").reset_index(drop=True)
    split_idx = int(len(window_df) * (1 - val_frac))

    train_part = window_df.iloc[:split_idx].copy().reset_index(drop=True)
    val_part = window_df.iloc[split_idx:].copy().reset_index(drop=True)

    return train_part, val_part

In [ ]:
def get_class_weights(y, pos_multiplier=1.5):
    classes = np.array([0, 1])
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {
        0: float(weights[0]),
        1: float(weights[1] * pos_multiplier)
    }

In [ ]:
def grid_search_on_window(window_df, eval_df, param_grid, val_frac=0.2, epochs=30, patience=4):
    train_part, val_part = split_train_val_last_20_percent(window_df, val_frac=val_frac)

    max_vals = get_max_values(train_part, val_part, eval_df)

    mu, sd = fit_num_scaler(train_part)
    train_scaled = apply_num_scaler(train_part, mu, sd)
    val_scaled = apply_num_scaler(val_part, mu, sd)

    X_train = get_keras_data(train_scaled)
    y_train = train_scaled[TARGET_COL].values.astype("float32")

    X_val = get_keras_data(val_scaled)
    y_val = val_scaled[TARGET_COL].values.astype("float32")

    class_weights = get_class_weights(y_train)

    pos_stats = pd.DataFrame([
        summarize_positive_counts("train", train_part),
        summarize_positive_counts("val", val_part),
        summarize_positive_counts("eval", eval_df)
    ])

    results = []
    best_score = -np.inf
    best_params = None

    for params in ParameterGrid(param_grid):
        tf.keras.backend.clear_session()
        gc.collect()

        model = build_embedding_nn(
            max_vals=max_vals,
            emb_n=params["emb_n"],
            dense_n=params["dense_n"],
            dropout=params["dropout"],
            lr=params["lr"]
        )

        es = EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=0
        )

        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=params["batch_size"],
            verbose=0,
            class_weight=class_weights,
            callbacks=[es]
        )

        val_pred = model.predict(X_val, batch_size=params["batch_size"], verbose=0).ravel()
        val_ap = average_precision_score(y_val, val_pred)

        results.append({
            "emb_n": params["emb_n"],
            "dense_n": params["dense_n"],
            "dropout": params["dropout"],
            "lr": params["lr"],
            "batch_size": params["batch_size"],
            "val_ap": float(val_ap)
        })

        if val_ap > best_score:
            best_score = val_ap
            best_params = params.copy()

        del model
        del history
        del val_pred
        gc.collect()
        tf.keras.backend.clear_session()

    results_df = pd.DataFrame(results).sort_values("val_ap", ascending=False).reset_index(drop=True)
    return best_params, best_score, results_df, train_part, val_part, max_vals, mu, sd, pos_stats

In [ ]:
def train_best_model(train_part, val_part, max_vals, best_params, mu, sd, epochs=30, patience=4):
    train_scaled = apply_num_scaler(train_part, mu, sd)
    val_scaled = apply_num_scaler(val_part, mu, sd)

    X_train = get_keras_data(train_scaled)
    y_train = train_scaled[TARGET_COL].values.astype("float32")

    X_val = get_keras_data(val_scaled)
    y_val = val_scaled[TARGET_COL].values.astype("float32")

    class_weights = get_class_weights(y_train)

    tf.keras.backend.clear_session()
    gc.collect()

    model = build_embedding_nn(
        max_vals=max_vals,
        emb_n=best_params["emb_n"],
        dense_n=best_params["dense_n"],
        dropout=best_params["dropout"],
        lr=best_params["lr"]
    )

    es = EarlyStopping(
        monitor="val_pr_auc",
        mode="max",
        patience=patience,
        restore_best_weights=True,
        verbose=0
    )

    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=best_params["batch_size"],
        verbose=0,
        class_weight=class_weights,
        callbacks=[es]
    )

    return model

In [ ]:
param_grid = {
    "emb_n": [16, 32],
    "dense_n": [128, 256],
    "dropout": [0.2],
    "lr": [0.001, 0.0005],
    "batch_size": [8192]
}

In [ ]:
pair = window_pairs[0]

train_A_full = prepare_window_data(df, pair["train_A_steps"])
train_B_full = prepare_window_data(df, pair["train_B_steps"])
eval_df = prepare_window_data(df, pair["eval_steps"])

best_params_A, best_score_A, grid_results_A, train_A_part, val_A_part, max_vals_A, mu_A, sd_A, pos_stats_A = grid_search_on_window(
    train_A_full,
    eval_df,
    param_grid=param_grid,
    val_frac=0.2,
    epochs=30,
    patience=4
)

best_model_A = train_best_model(
    train_A_part,
    val_A_part,
    max_vals_A,
    best_params_A,
    mu_A,
    sd_A,
    epochs=30,
    patience=4
)

best_params_B, best_score_B, grid_results_B, train_B_part, val_B_part, max_vals_B, mu_B, sd_B, pos_stats_B = grid_search_on_window(
    train_B_full,
    eval_df,
    param_grid=param_grid,
    val_frac=0.2,
    epochs=30,
    patience=4
)

best_model_B = train_best_model(
    train_B_part,
    val_B_part,
    max_vals_B,
    best_params_B,
    mu_B,
    sd_B,
    epochs=30,
    patience=4
)

print("Best A:", best_params_A, best_score_A)
print("Best B:", best_params_B, best_score_B)

display(pos_stats_A)
display(pos_stats_B)
display(grid_results_A.head())
display(grid_results_B.head())

Best A: {'batch_size': 8192, 'dense_n': 256, 'dropout': 0.2, 'emb_n': 32, 'lr': 0.0005} 0.6567034384717096
Best B: {'batch_size': 8192, 'dense_n': 256, 'dropout': 0.2, 'emb_n': 16, 'lr': 0.0005} 0.6291808880813041


,set,n_rows,n_pos,pos_rate
0,train,24800740,62137,0.002505
1,val,6200185,15167,0.002446
2,eval,10495401,18369,0.001750


,set,n_rows,n_pos,pos_rate
0,train,39357364,105974,0.002693
1,val,9839342,27819,0.002827
2,eval,10495401,18369,0.001750


,emb_n,dense_n,dropout,lr,batch_size,val_ap
0,32,256,0.2,0.0005,8192,0.656703
1,16,256,0.2,0.0005,8192,0.655689
2,16,256,0.2,0.0010,8192,0.652280
3,32,256,0.2,0.0010,8192,0.651490
4,32,128,0.2,0.0010,8192,0.649356


,emb_n,dense_n,dropout,lr,batch_size,val_ap
0,16,256,0.2,0.0005,8192,0.629181
1,32,256,0.2,0.0005,8192,0.626504
2,32,256,0.2,0.0010,8192,0.621950
3,16,256,0.2,0.0010,8192,0.617230
4,16,128,0.2,0.0005,8192,0.613211


In [ ]:
eval_A_scaled = apply_num_scaler(eval_df, mu_A, sd_A)
eval_B_scaled = apply_num_scaler(eval_df, mu_B, sd_B)

X_eval_A = get_keras_data(eval_A_scaled)
X_eval_B = get_keras_data(eval_B_scaled)
y_eval = eval_df[TARGET_COL].values.astype("float32")

eval_pred_A = best_model_A.predict(X_eval_A, batch_size=8192, verbose=0).ravel()
eval_pred_B = best_model_B.predict(X_eval_B, batch_size=8192, verbose=0).ravel()

pr_auc_A = average_precision_score(y_eval, eval_pred_A)
pr_auc_B = average_precision_score(y_eval, eval_pred_B)

print("Eval PR-AUC A:", pr_auc_A)
print("Eval PR-AUC B:", pr_auc_B)
display(pd.DataFrame([summarize_positive_counts("eval", eval_df)]))

Eval PR-AUC A: 0.49282058467463297
Eval PR-AUC B: 0.49819390822283527


,set,n_rows,n_pos,pos_rate
0,eval,10495401,18369,0.00175


In [ ]:
all_results = []

for pair in window_pairs:
    try:
        train_A_full = prepare_window_data(df, pair["train_A_steps"])
        train_B_full = prepare_window_data(df, pair["train_B_steps"])
        eval_df = prepare_window_data(df, pair["eval_steps"])

        best_params_A, best_score_A, grid_results_A, train_A_part, val_A_part, max_vals_A, mu_A, sd_A, pos_stats_A = grid_search_on_window(
            train_A_full,
            eval_df,
            param_grid=param_grid,
            val_frac=0.2,
            epochs=30,
            patience=4
        )

        best_model_A = train_best_model(
            train_A_part,
            val_A_part,
            max_vals_A,
            best_params_A,
            mu_A,
            sd_A,
            epochs=30,
            patience=4
        )

        best_params_B, best_score_B, grid_results_B, train_B_part, val_B_part, max_vals_B, mu_B, sd_B, pos_stats_B = grid_search_on_window(
            train_B_full,
            eval_df,
            param_grid=param_grid,
            val_frac=0.2,
            epochs=30,
            patience=4
        )

        best_model_B = train_best_model(
            train_B_part,
            val_B_part,
            max_vals_B,
            best_params_B,
            mu_B,
            sd_B,
            epochs=30,
            patience=4
        )

        eval_A_scaled = apply_num_scaler(eval_df, mu_A, sd_A)
        eval_B_scaled = apply_num_scaler(eval_df, mu_B, sd_B)

        X_eval_A = get_keras_data(eval_A_scaled)
        X_eval_B = get_keras_data(eval_B_scaled)
        y_eval = eval_df[TARGET_COL].values.astype("float32")

        eval_pred_A = best_model_A.predict(X_eval_A, batch_size=8192, verbose=0).ravel()
        eval_pred_B = best_model_B.predict(X_eval_B, batch_size=8192, verbose=0).ravel()

        pr_auc_A = average_precision_score(y_eval, eval_pred_A)
        pr_auc_B = average_precision_score(y_eval, eval_pred_B)

        eval_stats = summarize_positive_counts("eval", eval_df)

        all_results.append({
            "k": pair["k"],
            "train_A_steps": pair["train_A_steps"],
            "train_B_steps": pair["train_B_steps"],
            "eval_steps": pair["eval_steps"],
            "n_train_A": len(train_A_full),
            "n_train_B": len(train_B_full),
            "n_eval": len(eval_df),
            "n_pos_train_A": int(train_A_full[TARGET_COL].sum()),
            "n_pos_train_B": int(train_B_full[TARGET_COL].sum()),
            "n_pos_eval": eval_stats["n_pos"],
            "pos_rate_eval": eval_stats["pos_rate"],
            "best_score_A_val": best_score_A,
            "best_score_B_val": best_score_B,
            "pr_auc_A_eval": pr_auc_A,
            "pr_auc_B_eval": pr_auc_B,
            "best_params_A": str(best_params_A),
            "best_params_B": str(best_params_B)
        })

        del train_A_full, train_B_full, eval_df
        del train_A_part, val_A_part, train_B_part, val_B_part
        del best_model_A, best_model_B, grid_results_A, grid_results_B
        del X_eval_A, X_eval_B, y_eval, eval_pred_A, eval_pred_B
        del eval_A_scaled, eval_B_scaled, pos_stats_A, pos_stats_B
        gc.collect()
        tf.keras.backend.clear_session()

        print("Finished pair:", pair["k"])

    except Exception as e:
        print("Failed pair:", pair["k"], str(e))
        tf.keras.backend.clear_session()
        gc.collect()

results_df = pd.DataFrame(all_results)
display(results_df)

Finished pair: 2
Finished pair: 3
Finished pair: 4
Finished pair: 5
Finished pair: 6
Finished pair: 7
Finished pair: 8


,k,train_A_steps,train_B_steps,eval_steps,n_train_A,n_train_B,n_eval,n_pos_train_A,n_pos_train_B,n_pos_eval,pos_rate_eval,best_score_A_val,best_score_B_val,pr_auc_A_eval,pr_auc_B_eval,best_params_A,best_params_B
0,2,"[1, 2]","[2, 3]",[4],31000925,49196706,10495401,77304,133793,18369,0.001750,0.657079,0.623817,0.497598,0.479144,"{'batch_size': 8192, 'dense_n': 256, 'dropout'...","{'batch_size': 8192, 'dense_n': 256, 'dropout'..."
1,3,"[2, 3]","[3, 4]",[5],49196706,34936636,24633657,133793,86673,66514,0.002700,0.627197,0.512538,0.665880,0.650612,"{'batch_size': 8192, 'dense_n': 256, 'dropout'...","{'batch_size': 8192, 'dense_n': 256, 'dropout'..."
2,4,"[3, 4]","[4, 5]",[6],34936636,35129058,26489759,86673,84883,65412,0.002469,0.506507,0.715558,0.653480,0.647127,"{'batch_size': 8192, 'dense_n': 256, 'dropout'...","{'batch_size': 8192, 'dense_n': 256, 'dropout'..."
3,5,"[4, 5]","[5, 6]",[7],35129058,51123416,11747488,84883,131926,20781,0.001769,0.713447,0.652956,0.595563,0.558939,"{'batch_size': 8192, 'dense_n': 256, 'dropout'...","{'batch_size': 8192, 'dense_n': 256, 'dropout'..."
4,6,"[5, 6]","[6, 7]",[8],51123416,38237247,25540960,131926,86193,62828,0.002460,0.654245,0.548728,0.610912,0.600057,"{'batch_size': 8192, 'dense_n': 256, 'dropout'...","{'batch_size': 8192, 'dense_n': 256, 'dropout'..."
5,7,"[6, 7]","[7, 8]",[9],38237247,37288448,25997560,86193,83609,69056,0.002656,0.550193,0.608951,0.599324,0.576001,"{'batch_size': 8192, 'dense_n': 256, 'dropout'...","{'batch_size': 8192, 'dense_n': 256, 'dropout'..."
6,8,"[7, 8]","[8, 9]",[10],37288448,51538520,4556905,83609,131884,8278,0.001817,0.613707,0.621001,0.609807,0.607922,"{'batch_size': 8192, 'dense_n': 256, 'dropout'...","{'batch_size': 8192, 'dense_n': 256, 'dropout'..."
